# E4: SSS Marine Debris Detection — Balanced Dataset Training
**Dataset:** 609 images (160 positive + 449 diverse seabed backgrounds)
**Key fix from E3:** E3 had only 3 background images → model detected everything as debris. E4 has 449 diverse backgrounds so the model learns what "not debris" looks like.

In [ ]:
!pip install ultralytics pandas matplotlib -q

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload e4.zip
!unzip -q e4.zip -d /content/
!ls -la /content/e4/

In [ ]:
import os

print('=== E4 Dataset ===')
for split in ['train', 'val', 'test']:
    img_dir = f'/content/e4/images/{split}'
    lbl_dir = f'/content/e4/labels/{split}'
    n_img = len([f for f in os.listdir(img_dir) if f.endswith('.png')])
    n_pos = sum(1 for f in os.listdir(lbl_dir) if f.endswith('.txt') and os.path.getsize(os.path.join(lbl_dir, f)) > 0)
    n_neg = n_img - n_pos
    ratio = f'1:{n_neg/max(n_pos,1):.1f}' if n_pos > 0 else 'N/A'
    print(f'  {split}: {n_img} images ({n_pos} pos, {n_neg} neg) ratio={ratio}')

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Visualize: 4 positive + 4 background from train
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
train_imgs = sorted(os.listdir('/content/e4/images/train'))
pos_imgs = [f for f in train_imgs if 'BG' not in f]
neg_imgs = [f for f in train_imgs if 'BG' in f]

for i, img_name in enumerate(pos_imgs[:4] + neg_imgs[:4]):
    ax = axes[i // 4, i % 4]
    img = np.array(Image.open(f'/content/e4/images/train/{img_name}'))
    ax.imshow(img, cmap='gray')
    lbl_path = f'/content/e4/labels/train/{os.path.splitext(img_name)[0]}.txt'
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    _, xc, yc, w, h = map(float, parts)
                    rect = plt.Rectangle(((xc-w/2)*512, (yc-h/2)*512), w*512, h*512,
                                        fill=False, edgecolor='lime', linewidth=2)
                    ax.add_patch(rect)
        ax.set_title(img_name[:20], color='green', fontsize=8)
    else:
        ax.set_title('BACKGROUND', color='red', fontsize=8)
    ax.axis('off')
plt.suptitle('E4 Training: Green=debris bbox, Red=background (no box)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/e4/data_colab.yaml',
    epochs=150,
    imgsz=512,
    batch=16,
    patience=30,
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=3,
    # SSS-friendly: no flips, no rotation
    mosaic=0.0,
    mixup=0.0,
    fliplr=0.0,
    flipud=0.0,
    degrees=0.0,
    translate=0.05,
    scale=0.2,
    # Output
    name='e4_yolov8s',
    project='/content/runs',
    exist_ok=True,
    plots=True,
)

print('\n✓ Training complete!')
print('Best weights: /content/runs/e4_yolov8s/weights/best.pt')

In [ ]:
import pandas as pd

results_df = pd.read_csv('/content/runs/e4_yolov8s/results.csv')
results_df.columns = results_df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(results_df['epoch'], results_df['train/box_loss'], 'b-', label='Box')
axes[0].plot(results_df['epoch'], results_df['train/cls_loss'], 'r-', label='Cls')
axes[0].plot(results_df['epoch'], results_df['train/dfl_loss'], 'g-', label='DFL')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Training Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(results_df['epoch'], results_df['metrics/mAP50(B)'], 'b-', label='mAP50')
axes[1].plot(results_df['epoch'], results_df['metrics/mAP50-95(B)'], 'r-', label='mAP50-95')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP'); axes[1].set_title('Validation mAP')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(results_df['epoch'], results_df['metrics/precision(B)'], 'b-', label='Precision')
axes[2].plot(results_df['epoch'], results_df['metrics/recall(B)'], 'r-', label='Recall')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score'); axes[2].set_title('Precision & Recall')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

best_epoch = results_df['metrics/mAP50(B)'].idxmax()
print(f"Best epoch: {int(results_df.iloc[best_epoch]['epoch'])}")
print(f"Best mAP50: {results_df.iloc[best_epoch]['metrics/mAP50(B)']:.4f}")
print(f"Best mAP50-95: {results_df.iloc[best_epoch]['metrics/mAP50-95(B)']:.4f}")

In [ ]:
# Evaluate on test set (unseen data)
results_test = model.val(
    data='/content/e4/data_colab.yaml',
    split='test',
    imgsz=512,
    conf=0.25,
    save_json=True,
    workers=2,
)

print(f"\n=== TEST SET (UNSEEN DATA) ===")
print(f"mAP50:    {results_test.box.map50:.4f}")
print(f"mAP50-95: {results_test.box.map:.4f}")
print(f"Precision: {results_test.box.mp:.4f}")
print(f"Recall:    {results_test.box.mr:.4f}")
f1 = 2 * results_test.box.mp * results_test.box.mr / max(results_test.box.mp + results_test.box.mr, 1e-8)
print(f"F1:        {f1:.4f}")

In [ ]:
# Confidence threshold sweep — find optimal operating point
sweep_results = []
for conf in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7]:
    r = model.val(data='/content/e4/data_colab.yaml', split='val',
                  imgsz=512, conf=conf, verbose=False, workers=2)
    p, rv, m = r.box.mp, r.box.mr, r.box.map50
    f1 = 2 * p * rv / max(p + rv, 1e-8)
    sweep_results.append({'conf': conf, 'P': p, 'R': rv, 'mAP50': m, 'F1': f1})

sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

best = sweep_df.loc[sweep_df['F1'].idxmax()]
print(f"\nBest F1: conf={best['conf']:.2f}, F1={best['F1']:.4f}, P={best['P']:.4f}, R={best['R']:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df['conf'], sweep_df['P'], 'b-o', label='Precision')
ax.plot(sweep_df['conf'], sweep_df['R'], 'r-o', label='Recall')
ax.plot(sweep_df['conf'], sweep_df['F1'], 'g-o', label='F1', linewidth=2)
ax.axvline(best['conf'], color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision/Recall/F1 vs Confidence Threshold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/confidence_sweep.png', dpi=150)
plt.show()

In [ ]:
# Per-target detection analysis on test set
test_dir = '/content/e4/images/test'
test_lbl_dir = '/content/e4/labels/test'

target_files = {}
for f in sorted(os.listdir(test_dir)):
    if not f.endswith('.png'):
        continue
    parts = f.replace('.png', '').split('_')
    tid = 'BG'
    for p in parts:
        if p.startswith('TGT'):
            tid = p
            break
    if tid not in target_files:
        target_files[tid] = []
    target_files[tid].append(f)

pred_results = model.predict(source=test_dir, imgsz=512, conf=float(best['conf']),
                             save=False, verbose=False)

target_dets = {}
for r in pred_results:
    img_name = os.path.basename(str(r.path))
    parts = img_name.replace('.png', '').split('_')
    tid = 'BG'
    for p in parts:
        if p.startswith('TGT'):
            tid = p
            break
    if tid not in target_dets:
        target_dets[tid] = {'detected': 0, 'total': 0, 'confs': [], 'n_dets': []}
    target_dets[tid]['total'] += 1
    n_dets = len(r.boxes)
    target_dets[tid]['n_dets'].append(n_dets)
    if n_dets > 0:
        target_dets[tid]['detected'] += 1
        target_dets[tid]['confs'].extend([float(c) for c in r.boxes.conf])

print(f"{'Target':>10} {'Images':>7} {'Detected':>9} {'Rate':>8} {'Avg#':>6} {'AvgConf':>9}")
print('-' * 55)
for tid in sorted(target_dets.keys()):
    s = target_dets[tid]
    rate = s['detected'] / max(s['total'], 1)
    avg_c = np.mean(s['confs']) if s['confs'] else 0
    avg_n = np.mean(s['n_dets'])
    bar = '█' * int(rate * 20) + '░' * (20 - int(rate * 20))
    print(f"{tid:>10} {s['total']:>7} {s['detected']:>9} {bar} {rate:>7.0%} {avg_n:>5.1f} {avg_c:>8.3f}")

# KEY CHECK: BG should have LOW detection rate
bg_det = target_dets.get('BG', {}).get('detected', 0)
bg_tot = target_dets.get('BG', {}).get('total', 1)
print(f"\n⚠ Background false positive rate: {bg_det/bg_tot:.0%} ({bg_det}/{bg_tot})")
if bg_det / bg_tot > 0.1:
    print("  → HIGH false positives! Model still detecting background as debris.")
else:
    print("  → Good! Model is NOT detecting background as debris.")

In [ ]:
# Visual predictions on test images
pred_vis = model.predict(source=test_dir, imgsz=512, conf=float(best['conf']),
                         save=True, project='/content/runs', name='test_predictions_e4',
                         exist_ok=True, workers=2)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
pred_dir = '/content/runs/test_predictions_e4'
pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.jpg')])[:12]

for i, fname in enumerate(pred_files):
    ax = axes[i // 4, i % 4]
    img = np.array(Image.open(os.path.join(pred_dir, fname)))
    ax.imshow(img)
    ax.set_title(fname[:25], fontsize=8)
    ax.axis('off')
plt.suptitle('Test Set Predictions (green=debris, red=false positive)', fontsize=14)
plt.tight_layout()
plt.savefig('/content/test_predictions_grid.png', dpi=150)
plt.show()

In [ ]:
# Export to ONNX
model.export(format='onnx', imgsz=512)
print('Exported to ONNX')

# Download results
import shutil
shutil.make_archive('/content/e4_results', 'zip', '/content/runs/e4_yolov8s')
print('\nDownloads ready:')
print('  1. /content/runs/e4_yolov8s/weights/best.pt  — best model')
print('  2. /content/runs/e4_yolov8s/weights/last.pt  — last epoch')
print('  3. /content/runs/e4_yolov8s/weights/best.onnx — ONNX export')
print('  4. /content/e4_results.zip — full training results')
print('\nUse files.download() to save them to your machine.')